In [3]:
# ▶ 1. 필수 라이브러리 설치 및 임포트
!pip install openpyxl xlrd
import pandas as pd
import os
import re
import zipfile
from glob import glob

# ▶ 2. 압축 해제 (새로운 ZIP 파일 기준)
def unzip_file(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        print(f"✅ 압축 해제 완료: {extract_to}")

unzip_file("/content/14~23 주,야간 정리.zip", "/content/14_23")

# ▶ 3. 정제 함수 정의 (전처리/매핑 모두 동일하게 사용)
def clean_department_name(name):
    if pd.isna(name): return name
    name = str(name)
    name = re.sub(r"\(\d+\)", "", name)             # 숫자 괄호만 제거 (학과코드)
    name = re.sub(r"[・ㆍ·‧․.•∙･]", "", name)            # 가운뎃점 및 마침표 제거
    name = name.replace(" ", "")                    # 공백 제거
    return name.strip()

# ▶ 4. 연도별 계열 매핑 딕셔너리 생성
year_to_mapping = {}

for path in glob("/content/14_23/*.xls*"):
    try:
        df = pd.read_excel(path)
        if '학과명' in df.columns and '대계열' in df.columns and '조사년도' in df.columns:
            year = int(df['조사년도'].dropna().iloc[0])
            temp = df[['학과명', '대계열']].dropna().copy()
            temp['학과명'] = temp['학과명'].apply(clean_department_name)
            temp['대계열'] = temp['대계열'].astype(str).str.strip()
            mapping = dict(zip(temp['학과명'], temp['대계열']))

            if year in year_to_mapping:
                year_to_mapping[year].update(mapping)
            else:
                year_to_mapping[year] = mapping
    except Exception as e:
        print(f"❌ {os.path.basename(path)} → {e}")

# ▶ 5. 중도탈락 원본 엑셀 전처리
file_path = "/content/2020~2023 중도탈락.xlsx"  # 파일명만 바꾸면 재사용 가능
df_raw = pd.read_excel(file_path, header=[0, 1])  # 병합 셀 있는 헤더 처리

# ▶ 6. 멀티헤더 → 문자열 단일 헤더로 변환
df_raw.columns = [
    ' '.join([str(c).strip() for c in col if 'Unnamed' not in str(c)]).strip()
    for col in df_raw.columns
]
df = df_raw.copy()

# ▶ 7. 병합된 셀 해제 및 채우기
for col in ['기준연도', '학교','단과대학', '학과(전공)']:
    if col in df.columns:
        df[col] = df[col].ffill()
    else:
        print(f"⚠️ 열 없음: {col}")

# ▶ 8. 학과 정제 처리 (매핑과 일치하도록 통일 적용)
df['학과(전공)'] = df['학과(전공)'].apply(clean_department_name)

# ▶ 9. 사이버대학 / 원격 제거
if '학교종류' in df.columns:
    df = df[~df['학교종류'].str.contains("사이버대학", na=False)]
if '구분' in df.columns:
    df = df[~df['구분'].isin(['원격'])]

# ▶ 10. 계열 매핑
df['기준년도'] = pd.to_numeric(df['기준연도'], errors='coerce')
def match_by_year(row):
    year = int(row['기준년도']) if not pd.isna(row['기준년도']) else None
    dept = row['학과(전공)']
    mapping = year_to_mapping.get(year, {})
    return mapping.get(dept, None)

df['계열'] = df.apply(match_by_year, axis=1)
df['계열'] = df['계열'].fillna("미분류")

# ▶ 11. 결과 저장
output_name = "/content/2020~2023 중도탈락 계열분류.xlsx"
df.to_excel(output_name, index=False)
print(f"✅ 계열 분류 완료 → 저장 파일: {output_name}")


✅ 압축 해제 완료: /content/14_23
✅ 계열 분류 완료 → 저장 파일: /content/2020~2023 중도탈락 계열분류.xlsx


In [4]:
import pandas as pd

# ▶ 1. 파일 로딩
df = pd.read_excel("/content/2020~2023 중도탈락 계열분류.xlsx")

# ▶ 2. 자동 탐색
enroll_col = next((col for col in df.columns if "재적" in col and "학생" in col), None)
dropout_col = next((col for col in df.columns if "중도" in col and "탈락" in col and "계" in col), None)
old_rate_col = next((col for col in df.columns if "중도탈락학생비율" in col), None)

if not enroll_col or not dropout_col:
    raise ValueError("❌ 재적학생 또는 중도탈락학생 열이 존재하지 않습니다.")

# ▶ 3. 수치형 변환
df[enroll_col] = pd.to_numeric(df[enroll_col], errors='coerce')
df[dropout_col] = pd.to_numeric(df[dropout_col], errors='coerce')

# ▶ 4. 학과(전공) 자리에 계열 덮어쓰기
df['학과(전공)'] = df['계열']

# ▶ 5. 통합 기준
group_keys = ['기준연도', '학교', '학과(전공)']

# ▶ 6. 통합 규칙
agg_dict = {
    enroll_col: 'sum',
    dropout_col: 'sum'
}
for col in df.columns:
    if col not in group_keys + [enroll_col, dropout_col]:
        agg_dict[col] = 'first'

grouped = df.groupby(group_keys, as_index=False).agg(agg_dict)

# ▶ 7. 중도탈락률 계산
grouped['중도탈락률(%)'] = (grouped[dropout_col] / grouped[enroll_col]) * 100
grouped['중도탈락률(%)'] = grouped['중도탈락률(%)'].round(2)

# ▶ 8. 원본 컬럼 순서 기준 재배치
original_cols = pd.read_excel("/content/2020~2023 중도탈락 계열분류.xlsx", nrows=1).columns.tolist()

# ▶ 9. 기존 중도탈락학생비율 제거
if old_rate_col in original_cols:
    original_cols.remove(old_rate_col)

# ▶ 10. 중도탈락률(%)을 중도탈락 학생 계(B) 뒤에 삽입
if dropout_col in original_cols:
    insert_idx = original_cols.index(dropout_col) + 1
    final_cols = (
        original_cols[:insert_idx] +
        ['중도탈락률(%)'] +
        [col for col in original_cols[insert_idx:] if col in grouped.columns]
    )
else:
    final_cols = original_cols + ['중도탈락률(%)']

# ▶ 11. 순서 정리
df_final = grouped[final_cols]

# ▶ 12. 저장
output_path = "/content/2020~2023 중도탈락 최종계열통합.xlsx"
df_final.to_excel(output_path, index=False)
print(f"✅ 중도탈락률 위치까지 반영 완료 → {output_path}")


✅ 중도탈락률 위치까지 반영 완료 → /content/2020~2023 중도탈락 최종계열통합.xlsx
